# EMD4 senescence after exposure withdrawal

Run all cells to reproduce Figure 6. The simulation compares a model with feedback through secreted factors against reversible alternatives fitted during continuous arsenite exposure. The parameters are assigned and the simulation runs offline. [Installation and model notes](README.md).

In [ ]:
from pathlib import Path
import json
import subprocess
import sys
from tempfile import TemporaryDirectory

from IPython.display import Image, Markdown, display

here = Path.cwd().resolve()
root = next((path for path in (here, *here.parents, here / "EMD4")
             if (path / "src" / "emd4_simulation" / "run.py").is_file()), None)
if root is None:
    raise FileNotFoundError("Start Jupyter in the EMD4 directory or its parent.")
source_dir = root / "src"
if str(source_dir) not in sys.path:
    sys.path.insert(0, str(source_dir))
from emd4_simulation import FIGURE_FORMATS, FIGURE_STEM, remove_previous_figure

outdir = root / "outputs" / "reproduced"
outdir.mkdir(parents=True, exist_ok=True)

## Run the reference simulation

The analysis compares senescent burden and marker positivity after withdrawal on day 14. It fits reversible alternatives, estimates the dose at which the low-burden equilibrium disappears and follows the spatial spread of senescence. Parameter sensitivity is evaluated across 250 draws.

In [ ]:
summary_name = "emd4_simulation_summary.json"
tables_name = "emd4_manuscript_tables.json"
caption_name = f"{FIGURE_STEM}_legend.md"
figure_names = [f"{FIGURE_STEM}.{ext}" for ext in FIGURE_FORMATS]
artifacts = [summary_name, tables_name, caption_name, *figure_names]
log_path = outdir / "emd4_run.log"

print(f"Running the simulation. Log: {log_path.relative_to(root)}", flush=True)
with TemporaryDirectory(dir=outdir) as directory:
    stage = Path(directory)
    with log_path.open("w", encoding="utf-8") as log:
        for module in ("run", "manuscript_tables"):
            command = [sys.executable, "-u", "-m", f"emd4_simulation.{module}",
                       "--outdir", str(stage)]
            result = subprocess.run(command, cwd=source_dir, stdout=log, stderr=subprocess.STDOUT)
            if result.returncode:
                raise RuntimeError(f"The {module} step failed. See {log_path}.")

    summary = json.loads((stage / summary_name).read_text())
    tables = json.loads((stage / tables_name).read_text())
    checks = summary["checks"]
    assert tables["n_states"] == 14
    assert tables["n_checks"] == len(checks) == 13 and all(checks.values())
    assert summary["ensemble"]["n"] == 250
    assert summary["spatial"]["n_rep"] == 40
    for name in artifacts:
        path = stage / name
        if not path.is_file() or path.stat().st_size == 0:
            raise FileNotFoundError(f"Missing or empty output: {path}")
    for name in artifacts:
        (stage / name).replace(outdir / name)
    remove_previous_figure(outdir)

print(f"{sum(checks.values())}/{len(checks)} model checks passed. Results are in {outdir.relative_to(root)}/")

In [ ]:
display(Image(filename=str(outdir / f"{FIGURE_STEM}.png"), width=1100))
display(Markdown((outdir / caption_name).read_text(encoding="utf-8")))